# 01 - Kochi Metro induction data: exploratory checks
Runs on your private CSV if it is at `data/raw/kochi_induction_dataset.csv`, otherwise on the **synthetic** sample
(then every number below is meaningless; the notebook only demonstrates the checks).

The checks target assumptions the original notebook made silently: which columns leak the label, whether
`Standby` implies 0 km, whether the `Mileage_Category` thresholds fit the data, and how `Mileage_Label` / `Cleaning_Slot_Scheduled` are coded.

In [ ]:
import logging
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # notebooks/ -> repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

import pandas as pd
from src import config as cfg
from src.data.loader import load_raw_dataset, resolve_data_path

# Real data if data/raw/kochi_induction_dataset.csv exists, else the SYNTHETIC sample.
USE_SAMPLE = not cfg.RAW_DATA_PATH.exists()
path, SOURCE = resolve_data_path(use_sample=USE_SAMPLE)
df = load_raw_dataset(path)
print("data source:", SOURCE, "| shape:", df.shape)

## Schema and missing values

In [ ]:
from src.data.loader import RAW_COLUMNS
print("has the 19 notebook columns:", list(df.columns) == RAW_COLUMNS)
print(df.dtypes.to_string())
summary = pd.DataFrame({"n_missing": df.isna().sum(), "n_unique": df.nunique()})
summary

## Targets
`FinalDecision` (classification) and `Daily_Km_Assigned` (regression).

In [ ]:
print(df["FinalDecision"].value_counts(dropna=False))
print(df["Daily_Km_Assigned"].describe())

## Does `Standby` imply 0 km?
The original classifier cell left `Daily_Km_Assigned` in its features. If Standby always means 0 km, that column
leaks the label (removed in this repo). The same check decides whether `GATE_KM_BY_DECISION` is reasonable.

In [ ]:
by_decision = df.groupby("FinalDecision")["Daily_Km_Assigned"].agg(["count", "mean", "min", "max", lambda s: (s == 0).mean()])
by_decision.columns = ["count", "mean", "min", "max", "share_zero_km"]
by_decision

In [ ]:
from src.evaluation.visualization import plot_km_by_decision
plot_km_by_decision(df);

## Is the `Mileage_Category` feature informative?
The notebook hard-coded thresholds 120,000 / 200,000 on `Cum_Mileage`. If your `Cum_Mileage` never exceeds 120,000 the feature is constant.

In [ ]:
from src.data.preprocessing import encode_features
enc = encode_features(df)
print(df["Cum_Mileage"].describe())
print(enc["Mileage_Category"].value_counts(dropna=False))

## Categorical codings

In [ ]:
for col in ["Depot", "Mileage_Label", "Cleaning_Slot_Scheduled", "Stabling_Difficulty"]:
    print(col, "->", df[col].value_counts(dropna=False).to_dict())

## Which features relate to the label?
Rule-derived columns (e.g. validity flags) can make `FinalDecision` almost deterministic. This crosstab shows Service rate per flag value; inspect before trusting any accuracy number.

In [ ]:
flags = ["Rolling_Valid", "Signalling_Valid", "Telecom_Valid", "Any_Fitness_Invalid", "Critical_JobCard"]
service = (df["FinalDecision"] == "Service").astype(float)
pd.concat({f: service.groupby(df[f]).mean() for f in flags}, axis=1).round(3)

## Repeated trainsets
The notebook used a random row split. If the same `Trainset` appears on many dates, neighbouring days can land in train and test. Re-run training with `split_strategy="chronological"` to compare.

In [ ]:
print("unique trainsets:", df["Trainset"].nunique(), "| unique dates:", df["Date"].nunique(), "| rows:", len(df))